# День 3 — Baseline модель классификации

Цель: зафиксировать первую рабочую модель как точку отсчёта для всех следующих экспериментов.

Baseline: `TfidfVectorizer` по словам (униграммы) + `LogisticRegression`. Сплит стратифицированный, 75/25, `random_state=42`.

Что проверяется:
- vectorization и обучение модели;
- предсказания на test set и новых текстах;
- метрики `accuracy`, `f1_macro`, `f1_weighted`;
- сохранённые артефакты: модель, метрики, отчёт, предсказания, строка в `model_comparison.csv`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report

sys.path.insert(0, str(Path.cwd().parent / "src"))

from classification.config import (
    BASELINE_METRICS_PATH,
    BASELINE_MODEL_PATH,
    BASELINE_PREDICTIONS_PATH,
    BASELINE_REPORT_PATH,
    COMPARISON_PATH,
    LABEL_COL,
    TEXT_COL,
)
from classification.load_data import split_data
from classification.predict import predict_texts
from classification.preprocess import get_training_data
from classification.train_baseline import build_baseline, evaluate

df = get_training_data()
train_df, test_df = split_data(df)
print(f"Строк: {len(df)}, train: {len(train_df)}, test: {len(test_df)}")
display(
    pd.DataFrame(
        {
            "train": train_df[LABEL_COL].value_counts(),
            "test": test_df[LABEL_COL].value_counts(),
        }
    ).sort_index()
)

Строк: 8839, train: 6629, test: 2210


,train,test
label,,
negative,1978,660
neutral,3203,1068
positive,1448,482


## 1. Обучение baseline

In [2]:
model = build_baseline()
model.fit(train_df[TEXT_COL], train_df[LABEL_COL])

y_pred = model.predict(test_df[TEXT_COL])
metrics = evaluate(test_df[LABEL_COL], y_pred)
print("Метрики на test:", metrics)

report = classification_report(test_df[LABEL_COL], y_pred, output_dict=True, zero_division=0)
display(pd.DataFrame(report).T)

Метрики на test: {'accuracy': 0.5995, 'f1_macro': 0.5389, 'f1_weighted': 0.5765}


,precision,recall,f1-score,support
negative,0.571992,0.439394,0.497001,660.000000
neutral,0.594903,0.830524,0.693240,1068.000000
positive,0.698113,0.307054,0.426513,482.000000
accuracy,0.599548,0.599548,0.599548,0.599548
macro avg,0.621669,0.525657,0.538918,2210.000000
weighted avg,0.610571,0.599548,0.576461,2210.000000


## 2. Признаки TF-IDF

In [3]:
tfidf = model.named_steps["tfidf"]
classifier = model.named_steps["clf"]
features = tfidf.get_feature_names_out()
print(f"TF-IDF признаков: {len(features)}")
print(f"Первые признаки: {list(features[:15])}")

coefficients = pd.DataFrame(
    classifier.coef_,
    index=classifier.classes_,
    columns=features,
)
for label in classifier.classes_:
    top = coefficients.loc[label].sort_values(ascending=False).head(5)
    print(f"{label}: {', '.join(top.index)}")

TF-IDF признаков: 15457
Первые признаки: ['00', '01', '02', '022', '03', '05', '06', '07', '08', '09', '096', '0_о', '0x800a03ec', '10', '100']
negative: блять, пиздец, нахуй, сука, ненавижу
neutral: про, хочешь, или, из, вопрос
positive: люблю, вау, нравится, рада, хорошо


## 3. Предсказания

In [4]:
demo_texts = [
    "это просто космос, очень понравилось!",
    "ужасно, не рекомендую",
    "вроде нормально, ничего особенного",
]
display(predict_texts(demo_texts, model=model))

prediction_sample = pd.DataFrame(
    {
        "text": test_df[TEXT_COL].head(5),
        "true": test_df[LABEL_COL].head(5),
        "pred": y_pred[:5],
    }
)
display(prediction_sample)

,text,label,confidence
0,"это просто космос, очень понравилось!",positive,0.6011
1,"ужасно, не рекомендую",negative,0.6521
2,"вроде нормально, ничего особенного",neutral,0.6071


,text,true,pred
0,это сын позднякова или как,neutral,neutral
1,хорошо главное не выгори,positive,positive
2,это все ретроградный меркурий,neutral,positive
3,это тоже можно,neutral,neutral
4,сотрудники росгвардии спасли упавшего в москву...,neutral,neutral


## 4. Сохранённые результаты и точка отсчёта

In [5]:
saved_metrics = json.loads(BASELINE_METRICS_PATH.read_text(encoding="utf-8"))
saved_report = json.loads(BASELINE_REPORT_PATH.read_text(encoding="utf-8"))
saved_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH)
comparison = pd.read_csv(COMPARISON_PATH)

print("Метрики совпадают с пересчитанными:", metrics == saved_metrics)
print("Файлы артефактов exist:", {
    "model": BASELINE_MODEL_PATH.exists(),
    "metrics": BASELINE_METRICS_PATH.exists(),
    "report": BASELINE_REPORT_PATH.exists(),
    "predictions": BASELINE_PREDICTIONS_PATH.exists(),
})
print(f"Сохранено предсказаний: {len(saved_predictions)}")
display(pd.Series(saved_report["split"], name="value").to_frame())
display(comparison[comparison["model"] == "baseline_tfidf_logreg"])

Метрики совпадают с пересчитанными: True
Файлы артефактов exist: {'model': True, 'metrics': True, 'report': True, 'predictions': True}
Сохранено предсказаний: 2210


,value
type,stratified
test_size,0.25
random_state,42
train_rows,6629
test_rows,2210


,model,val_accuracy,val_f1_macro,test_accuracy,test_f1_macro,test_f1_weighted,selected
0,baseline_tfidf_logreg,0.608,0.5545,0.5995,0.5389,0.5765,False


## Выводы

- Baseline обучен на 6629 текстах и оценён на 2210 текстах test set.
- Метрики: accuracy `0.5995`, f1_macro `0.5389`, f1_weighted `0.5765`.
- Слабое место — классы `negative` и `positive`: модель часто предсказывает `neutral`; это ожидаемая точка роста для Дня 4.
- Результат зафиксирован в `baseline.joblib`, `baseline_metrics.json`, `baseline_report.json`, `baseline_predictions.csv` и в строке `baseline_tfidf_logreg` таблицы `model_comparison.csv`.